# DEMO - Testing and Training HerdNet on nadir aerial images

## Installations

In [ ]:
# Check if GPU is available
!nvidia-smi

In [ ]:
# pip install wandb --upgrade
!pip install --upgrade --force-reinstall typing_extensions

In [ ]:
## If an old GPU like GTX 1080 is available install older torch version which is compatible
# !pip uninstall -y torch torchvision torchaudio
#
#!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118

In [ ]:

!pip install ipywidgets
!pip install huggingface-hub


# !unzip -oq ./data.zip -d ./

In [ ]:
# Download and install the code
import sys

!git clone -b dinov3 https://github.com/cwinkelmann/HerdNet

!pip install ./HerdNet

sys.path.append('./HerdNet')

## Download Models and Demo Datasets

In [ ]:
from huggingface_hub import hf_hub_download
from pathlib import Path
local_dir = Path("./models").resolve()

# Download specific files
model_path = hf_hub_download(
    repo_id="karisu/HerdNet",
    filename="general_2022/model.pth",  # adjust filename as needed
    local_dir=local_dir
)

config_path = hf_hub_download(
    repo_id="karisu/HerdNet",
    filename="general_2022/config.yaml",
    local_dir=local_dir
)

print(f"Model Path: {model_path} and Config Path: {config_path}")

In [ ]:
path = hf_hub_download(
    repo_id="karisu/General_Dataset",
    repo_type="dataset",
    filename="general_dataset.zip",  # specify the file path within the repo
    local_dir="./data",
)

In [ ]:
# Install unzip on a linux system, unzip manually otherwise 
!apt-get update && apt-get install -y unzip



In [ ]:
!unzip -oq ./data/general_dataset.zip -d ./data

In [ ]:
# Test installation
# Set the seed
from animaloc.utils.seed import set_seed

set_seed(9292)

In [ ]:
# Training, validation and test datasets
import albumentations as A

from animaloc.datasets import CSVDataset
from animaloc.data.transforms import MultiTransformsWrapper, DownSample, PointsToMask, FIDT

patch_size = 512
num_classes = 7
down_ratio = 2

train_dataset = CSVDataset(
    csv_file = './data/train_patches.csv',
    root_dir = './data/train_patches',
    albu_transforms = [
        A.VerticalFlip(p=0.5),
        A.HorizontalFlip(p=0.5),
        A.RandomRotate90(p=0.5),
        A.RandomBrightnessContrast(brightness_limit=0.2, contrast_limit=0.2, p=0.2),
        A.Blur(blur_limit=15, p=0.2),
        A.Normalize(p=1.0)
        ],
    end_transforms = [MultiTransformsWrapper([
        FIDT(num_classes=num_classes, down_ratio=down_ratio),
        PointsToMask(radius=2, num_classes=num_classes, squeeze=True, down_ratio=int(patch_size//16))
        ])]
    )

val_dataset = CSVDataset(
    csv_file = './data/val_patches/gt.csv',
    root_dir = './data/val_patches',
    albu_transforms = [A.Normalize(p=1.0)],
    end_transforms = [DownSample(down_ratio=down_ratio, anno_type='point')]
    )

test_dataset = CSVDataset(
    csv_file = './data/test.csv',
    root_dir = './data/test',
    albu_transforms = [A.Normalize(p=1.0)],
    end_transforms = [DownSample(down_ratio=down_ratio, anno_type='point')]
    )

In [ ]:
# Dataloaders
from torch.utils.data import DataLoader

train_dataloader = DataLoader(dataset = train_dataset, batch_size = 12, shuffle = True)

val_dataloader = DataLoader(dataset = val_dataset, batch_size = 1, shuffle = False)

test_dataloader = DataLoader(dataset = test_dataset, batch_size = 1, shuffle = False)

## Define HerdNet for training

In [ ]:
from animaloc.models import HerdNet
from torch import Tensor
from animaloc.models import LossWrapper
from animaloc.train.losses import FocalLoss
from torch.nn import CrossEntropyLoss

herdnet = HerdNet(num_classes=num_classes, down_ratio=down_ratio).cuda()

weight = Tensor([0.1, 1.0, 2.0, 1.0, 6.0, 12.0, 1.0]).cuda()

losses = [
    {'loss': FocalLoss(reduction='mean'), 'idx': 0, 'idy': 0, 'lambda': 1.0, 'name': 'focal_loss'},
    {'loss': CrossEntropyLoss(reduction='mean', weight=weight), 'idx': 1, 'idy': 1, 'lambda': 1.0, 'name': 'ce_loss'}
    ]

## This loss Wrapper is important for the Evaluator
herdnet = LossWrapper(herdnet, losses=losses)

## Create the Trainer

In [ ]:
from torch.optim import Adam
from animaloc.train import Trainer
from animaloc.eval import PointsMetrics, HerdNetStitcher, HerdNetEvaluator
from animaloc.utils.useful_funcs import mkdir

work_dir = './output'
mkdir(work_dir)

lr = 1e-4
weight_decay = 1e-3
epochs = 5

optimizer = Adam(params=herdnet.parameters(), lr=lr, weight_decay=weight_decay)

metrics = PointsMetrics(radius=20, num_classes=num_classes)

stitcher = HerdNetStitcher(
    model=herdnet,
    size=(patch_size,patch_size),
    overlap=160,
    down_ratio=down_ratio,
    reduction='mean'
    )

evaluator = HerdNetEvaluator(
    model=herdnet,
    dataloader=val_dataloader,
    metrics=metrics,
    stitcher=stitcher,
    work_dir=work_dir,
    header='validation'
    )

trainer = Trainer(
    model=herdnet,
    train_dataloader=train_dataloader,
    optimizer=optimizer,
    num_epochs=epochs,
    evaluator=evaluator,
    work_dir=work_dir
    )

## Test the pretrained model

In [ ]:

# Load trained parameters
from animaloc.models import load_model
pth_path = './models/general_2022/model.pth'

herdnet = load_model(herdnet, pth_path=pth_path)
herdnet = LossWrapper(herdnet, losses=losses)

In [ ]:
# Create output folder
test_dir = './test_output'
mkdir(test_dir)

In [ ]:
# Create an Evaluator
test_evaluator = HerdNetEvaluator(
    model=herdnet,
    dataloader=test_dataloader,
    metrics=metrics,
    stitcher=stitcher,
    work_dir=test_dir,
    header='test'
    )

In [ ]:
# Start testing
test_f1_score = test_evaluator.evaluate(returns='f1_score')

In [ ]:
# Print global F1 score (%)
print(f"F1 score = {test_f1_score * 100:0.0f}%")
# New implementation training 38%
# loaded model 83%

In [ ]:
# Get the detections
detections = test_evaluator.results
detections

In [ ]:
## Load the default model

In [ ]:
test_f1_score = test_evaluator.evaluate(returns='f1_score')

In [ ]:
# Print global F1 score (%)
print(f"F1 score = {test_f1_score * 100:0.0f}%")
# New implementation 38%
# Old Tag 0.2.1

## Actuall Training

### Manual Patch creation
Patch larger images into trainable tiles

In [ ]:
# Create validation patches using the patcher tool (for demo)
from animaloc.utils.useful_funcs import mkdir

mkdir('./data/val_patches')
!python ./HerdNet/tools/patcher.py ./data/val 512 512 0 ./data/val_patches -csv ./data/val.csv -min 0.0 -all False

In [ ]:
trainer.start(warmup_iters=100, checkpoints='best', select='max', validate_on='f1_score')

In [ ]:
# Create an Evaluator
test_evaluator = HerdNetEvaluator(
    model=herdnet_general,
    dataloader=test_dataloader,
    metrics=metrics,
    stitcher=stitcher,
    work_dir=test_dir,
    header='test'
    )